# Seven publication figures

This notebook builds exactly seven test-only publication figure groups from completed runs. Every metric figure is restricted to hierarchy loss `h=0`, loss weights `genus=1.0, species=0.5, age=2.0`, and the 30 spaced seeds from `40` through `2940`. Validation, development-withheld, other weight recipes, other hierarchy settings, and other seeds are excluded:

1. test macro-F1 for every prediction task and model;
2. a ConvNeXt visual-ablation figure with Gaussian blur, resolution loss, colour retention, patch shuffling, and one larger interaction panel;
3. a detailed ConvNeXt-Base single-species data ablation versus its matched full-data baseline; and
4. every ConvNeXt-Base species-stage data ablation versus its matched baseline, with a separate shared-SD effect for each species-stage row;
5. raw recall-margin decomposition for the selected species, with Adult and Juvenile labeled separately; and
6. raw recall-margin decomposition for every species-stage row; and
7. ten representative transformations applied to five reproducibly sampled test worms.

Figures 3 and 4 show target-class recall only. They use the full-data model's sample standard deviation as one shared unit: `d_total = (normal − chance) / s_normal`, `d_ablation = (normal − ablated) / s_normal`, and `d_retained = (ablated − chance) / s_normal`, so `d_total = d_ablation + d_retained`. Figures 5 and 6 show the matching unstandardised margins: `M_total = normal − chance`, `M_lost = normal − ablated`, and `M_retained = ablated − chance`. Chance is derived as `1/K` from each task's saved class map under uniform random prediction.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

PAPER_ROOT = PROJECT_ROOT / 'publication_30seed_result'
TAXON_STAGE_ROOT = PAPER_ROOT
OUTPUT_DIR = PAPER_ROOT / 'publication_bundle' / 'figures'
DATA_ROOT = PROJECT_ROOT.parent / 'petridish-worm-images'
PAPER_ROOT, TAXON_STAGE_ROOT, OUTPUT_DIR


## Figure settings

Change these values to inspect a different Figure 2 visual backbone or the species used by Figures 3 and 5. Figures 3 through 6 are fixed to ConvNeXt-Base.


In [ ]:
VISUAL_MODEL = 'convnext_base'
SPECIES_ABLATION = 'Aporrectodea_longa'


## Build the seven figures

Only completed test results are used. Figures 1–2 read `test_*` metrics. Figures 3–6 use only `convnext_base`, target recall, and `cohort == 'independent_test'`; Figure 7 samples only the fixed test split. Other models, macro-F1, validation, and development-withheld cohorts are excluded from Figures 3–6. Every figure is saved as PNG, PDF, and SVG together with its plotted CSV data and a manifest.


In [ ]:
import importlib
from scripts import build_holdout_visual_notebook as figure_builder
importlib.reload(figure_builder)

manifest = figure_builder.build_holdout_visual_notebook_figures(
    paper_root=PAPER_ROOT,
    output_dir=OUTPUT_DIR,
    taxon_stage_root=TAXON_STAGE_ROOT,
    visual_model=VISUAL_MODEL,
    species_ablation=SPECIES_ABLATION,
    split_root=PROJECT_ROOT,
    data_root=DATA_ROOT,
)
manifest


## Display the seven figures


In [ ]:
from IPython.display import Image as DisplayImage, display

FIGURE_STEMS = [
    'figure_01_all_models_all_tasks',
    'figure_02_convnext_visual_ablation',
    'figure_03_species_ablation',
    'figure_04_all_data_ablations',
    'figure_05_species_ablation_raw_margins',
    'figure_06_all_data_ablations_raw_margins',
    'figure_07_representative_transformations',
]
for stem in FIGURE_STEMS:
    path = OUTPUT_DIR / f'{stem}.png'
    if path.exists():
        print(path.name)
        display(DisplayImage(filename=str(path)))
    else:
        print(f'Not created (no completed input rows): {path.name}')


## Read the distribution-aware effects

In Figure 3, ConvNeXt-Base is the only model shown and Adult and Juvenile are separate labeled rows. For each task, chance is fixed at `0` shared-SD units, the orange point is the ablated capacity retained (`d_retained`), the hollow blue diamond is total normal-model capacity (`d_total`), and the red distance between them is capacity lost to ablation (`d_ablation`). The three values are printed on the plot and are exactly additive.

Figure 4 applies the same visual grammar to every ConvNeXt-Base species-stage holdout; `Adult:` and `Juvenile:` remain explicit in every row label. Figures 5 and 6 repeat the selected-species and all-species views in raw target-recall margin units: chance is zero, the orange point is `M_retained`, the hollow blue diamond is `M_total`, and the red gap is `M_lost`. These plots contain no metric table, macro-F1, ratio, bootstrap interval, or p-value. The publication pipeline uses 30 spaced seeds (40 through 2940). If the normal-model seed recalls have zero variance, the standardised effects are undefined, while the raw-margin figures remain defined. Figure 7 shows the same representative transformations for five reproducibly sampled test worms.


In [ ]:
import pandas as pd

effect_files = {
    **{stem: OUTPUT_DIR / 'figure_sources' / stem / 'shared_variance_effects.csv' for stem in FIGURE_STEMS[2:4]},
    **{stem: OUTPUT_DIR / 'figure_sources' / stem / 'raw_margin_decomposition.csv' for stem in FIGURE_STEMS[4:6]},
}
effect_summaries = {
    stem: pd.read_csv(path)
    for stem, path in effect_files.items()
    if path.exists()
}
effect_summaries
